# 05 - Recommendations
## NYC Taxi Revenue Optimization Project

**Business Question:**
"Given historical trip data, which zones, hours, and route types
should a NYC taxi fleet prioritize to maximize revenue per driver per shift?"

**Data:** 237.7M cleaned trips — 2019, 2022–2025
**Analysis basis:** 2023–2025 post-fare-restructure data

**This notebook:**
1. Summarizes key findings into actionable recommendations
2. Builds visualizations to support each recommendation
3. Produces a final summary table for fleet managers

### Before the Recommendations — Market Context

- NYC taxi market is at 48% of pre-COVID volume
- Revenue per trip stabilized at ~$28.65–$28.90 since 2023 fare restructure
- Tip rates declining year over year (20.81% in 2022 → 17.94% in 2025)
- 2025 is strongest post-COVID year — market recovering

**Implication:** Higher per-trip revenue compensates for lower volume.
A driver working smart zones and hours can earn well
despite the smaller market.

In [0]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#2e3245",
    "axes.labelcolor":  "#c9cde0",
    "xtick.color":      "#7a7f99",
    "ytick.color":      "#7a7f99",
    "text.color":       "#c9cde0",
    "grid.color":       "#2e3245",
    "grid.linewidth":   0.6,
    "font.family":      "monospace",
    "axes.titlesize":   13,
    "axes.labelsize":   10,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
})

GOLD  = "#f5c842"
TEAL  = "#3ecfcf"
CORAL = "#ff6b6b"
MUTED = "#5a5f7a"
WHITE = "#e8eaf0"
BG    = "#0f1117"

print("✅ Style config loaded")

### Visualization 1 — Revenue Per Trip by Year

Tracks the fare restructure impact across 2019–2025.
2023 is the inflection point — everything before it is a different market.

In [0]:
years  = [2019, 2022, 2023, 2024, 2025]
rev    = [21.62, 21.73, 28.90, 28.65, 28.74]
colors = [MUTED, MUTED, GOLD, GOLD, GOLD]

fig, ax = plt.subplots(figsize=(9, 4.5))

bars = ax.bar(years, rev, color=colors, width=0.55, zorder=3)

for bar, v in zip(bars, rev):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.2,
            f"${v:.2f}", ha="center", va="bottom", fontsize=9, color=WHITE)

ax.axhline(28.65, color=TEAL, linewidth=1, linestyle="--", alpha=0.6)
ax.text(2024.6, 27.9, "floor — $28.65", color=TEAL, fontsize=8, ha="center")

ax.set_xticks(years)
ax.set_ylabel("Avg Revenue / Trip ($)")
ax.set_title("Revenue Per Trip by Year — Fare Restructure Impact")
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
ax.set_ylim(0, 33)

pre  = mpatches.Patch(color=MUTED, label="Pre-restructure (2019–2022)")
post = mpatches.Patch(color=GOLD,  label="Post-restructure (2023–2025)")
ax.legend(handles=[pre, post], fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

### Visualization 2 — Revenue Per Minute by Hour of Day

Efficiency metric — isolates the hours where a driver earns most per minute on the clock.
4am–6am and 6pm–8pm are the two peak windows.

In [0]:
hours   = list(range(24))
rev_min = [
    2.37, 2.48, 2.49, 2.53, 2.57, 2.55,  # 0–5
    2.28, 2.18, 2.15, 2.10, 2.08, 2.07,  # 6–11
    2.09, 2.11, 2.14, 2.18, 2.22, 2.31,  # 12–17
    2.38, 2.42, 2.41, 2.39, 2.36, 2.33,  # 18–23
]

peak = [v if (3 <= i <= 5 or 18 <= i <= 20) else np.nan for i, v in enumerate(rev_min)]

fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(hours, rev_min, alpha=0.12, color=TEAL)
ax.plot(hours, rev_min, color=TEAL, linewidth=2.5, zorder=3)
ax.scatter(hours, peak, color=GOLD, s=70, zorder=5, label="Peak window")

ax.annotate("4am–6am\n$2.55–$2.57/min", xy=(4, 2.57),
            xytext=(7, 2.60), color=GOLD, fontsize=8.5,
            arrowprops=dict(arrowstyle="->", color=GOLD, lw=0.8))

ax.annotate("6pm–8pm\n$2.38–$2.42/min", xy=(19, 2.42),
            xytext=(16, 2.53), color=GOLD, fontsize=8.5,
            arrowprops=dict(arrowstyle="->", color=GOLD, lw=0.8))

ax.set_xticks(hours)
ax.set_xticklabels([f"{h:02d}:00" for h in hours], rotation=45, ha="right")
ax.set_ylabel("Avg Revenue / Minute ($)")
ax.set_title("Revenue Per Minute by Hour of Day — 2023–2025")
ax.set_ylim(1.95, 2.75)   # ← tight range — makes the curve variation readable
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
ax.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

### Visualization 3 — Tip Percentage by Hour of Day

Tip rate varies meaningfully by hour — 5pm–7pm peaks at 21%+.
Drivers working evening rush capture both high volume and high tips.

In [0]:
tip_pct = [
    18.92, 18.45, 18.21, 18.10, 17.98, 18.33,  # 0–5
    18.67, 19.12, 19.45, 19.78, 20.11, 20.34,  # 6–11
    20.21, 20.15, 20.08, 20.42, 20.88, 21.12,  # 12–17
    21.05, 20.87, 20.43, 19.98, 19.54, 19.21,  # 18–23
]

peak_tip = [v if (16 <= i <= 18) else np.nan for i, v in enumerate(tip_pct)]

fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(hours, tip_pct, alpha=0.12, color=CORAL)
ax.plot(hours, tip_pct, color=CORAL, linewidth=2.5, zorder=3)
ax.scatter(hours, peak_tip, color=GOLD, s=70, zorder=5, label="Peak tip window")

ax.annotate("5pm–7pm\n21%+ tip rate", xy=(17, 21.12),
            xytext=(19, 21.35), color=GOLD, fontsize=8.5,
            arrowprops=dict(arrowstyle="->", color=GOLD, lw=0.8))

ax.set_xticks(hours)
ax.set_xticklabels([f"{h:02d}:00" for h in hours], rotation=45, ha="right")
ax.set_ylabel("Avg Tip %")
ax.set_title("Tip Percentage by Hour of Day — 2023–2025")
ax.set_ylim(17.0, 22.0)
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
ax.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

### Visualization 4 — Top 10 Pickup Zones by Avg Revenue Per Trip

Airport zones operate in a completely different revenue tier.
JFK and LaGuardia are not outliers — they are the strategy.

In [0]:
zones = [
    "Newark (1)", "JFK (132)", "LaGuardia (138)",
    "East Harlem (70)", "Midtown (161)", "Murray Hill (162)",
    "Upper East Side (236)", "Midtown (163)", "Sutton (234)", "Clinton (140)"
]
avg_rev = [100.83, 71.69, 57.03, 62.45, 34.21, 33.87, 32.14, 31.98, 31.45, 30.92]
colors  = [GOLD, GOLD, GOLD, TEAL, MUTED, MUTED, MUTED, MUTED, MUTED, MUTED]

# sort ascending for horizontal bar readability
sorted_pairs = sorted(zip(avg_rev, zones, colors))
avg_rev_s, zones_s, colors_s = zip(*sorted_pairs)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(zones_s, avg_rev_s, color=colors_s, height=0.6, zorder=3)

for bar, v in zip(bars, avg_rev_s):
    ax.text(v + 0.5, bar.get_y() + bar.get_height() / 2,
            f"${v:.2f}", va="center", fontsize=8.5, color=WHITE)

ax.set_xlabel("Avg Revenue / Trip ($)")
ax.set_title("Top 10 Pickup Zones by Avg Revenue Per Trip — 2023–2025")
ax.xaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
ax.set_xlim(0, 115)

airport = mpatches.Patch(color=GOLD,  label="Airport zone")
city    = mpatches.Patch(color=TEAL,  label="Top city zone")
other   = mpatches.Patch(color=MUTED, label="Other city zone")
ax.legend(handles=[airport, city, other], fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

### Visualization 5 — Revenue by Route Type

Airport routes generate 3.7x more revenue per trip than city-to-city.
LaGuardia returns 83% deadhead value vs JFK's 51% — positioning matters.

In [0]:
route_types = ["City → City", "City → Outer\nBorough", "Outer Borough\n→ City", "City → Airport", "Airport → City"]
avg_rev_r   = [19.84, 22.41, 24.17, 73.52, 58.63]
avg_min_r   = [11.2,  14.8,  16.1,  38.4,  31.2]
colors_r    = [MUTED, MUTED, MUTED, GOLD, TEAL]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Avg Revenue Per Trip ─────────────────────────────────────────────
bars1 = axes[0].bar(route_types, avg_rev_r, color=colors_r, width=0.55, zorder=3)
for bar, v in zip(bars1, avg_rev_r):
    axes[0].text(bar.get_x() + bar.get_width() / 2, v + 0.5,
                 f"${v:.2f}", ha="center", va="bottom", fontsize=8.5, color=WHITE)

axes[0].set_ylabel("Avg Revenue / Trip ($)")
axes[0].set_title("Avg Revenue Per Trip by Route Type")
axes[0].yaxis.grid(True, zorder=0)
axes[0].set_axisbelow(True)
axes[0].set_ylim(0, 88)
axes[0].tick_params(axis="x", labelsize=8)

# ── Right: Avg Trip Duration ───────────────────────────────────────────────
bars2 = axes[1].bar(route_types, avg_min_r, color=colors_r, width=0.55, zorder=3)
for bar, v in zip(bars2, avg_min_r):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.3,
                 f"{v} min", ha="center", va="bottom", fontsize=8.5, color=WHITE)

axes[1].set_ylabel("Avg Trip Duration (min)")
axes[1].set_title("Avg Trip Duration by Route Type")
axes[1].yaxis.grid(True, zorder=0)
axes[1].set_axisbelow(True)
axes[1].set_ylim(0, 46)
axes[1].tick_params(axis="x", labelsize=8)

airport = mpatches.Patch(color=GOLD,  label="City → Airport")
return_  = mpatches.Patch(color=TEAL,  label="Airport → City")
city    = mpatches.Patch(color=MUTED, label="City routes")
fig.legend(handles=[airport, return_, city], fontsize=8,
           loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.04))

plt.suptitle("Route Type Analysis — 2023–2025", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Visualization 6 — Zone × Hour Revenue Heatmap

Best zone + hour combinations by avg revenue per trip.
JFK afternoon is the clear outlier — $84–$89/trip in a 5-hour window.

In [0]:
zone_labels = [
    "Newark (1)", "JFK (132)", "LaGuardia (138)",
    "East Harlem (70)", "Midtown (161)", "Sutton (234)"
]

hour_labels = ["6am–9am", "9am–12pm", "12pm–2pm", "2pm–7pm", "7pm–10pm", "10pm–2am"]

# rows = zones, cols = hour windows
data = np.array([
    [98.2,  99.1,  100.4, 101.2, 99.8,  102.1],  # Newark
    [71.2,  73.4,  76.8,  88.5,  84.1,  79.3 ],  # JFK
    [55.1,  56.8,  57.4,  59.2,  58.6,  57.8 ],  # LaGuardia
    [60.2,  61.4,  63.1,  67.8,  69.4,  62.1 ],  # East Harlem
    [31.2,  32.8,  33.4,  35.6,  36.1,  34.2 ],  # Midtown
    [29.8,  30.4,  31.1,  33.2,  34.8,  32.1 ],  # Sutton
])

fig, ax = plt.subplots(figsize=(11, 6))

im = ax.imshow(data, cmap="YlOrBr", aspect="auto")

# ── Cell annotations ──────────────────────────────────────────────────────
for i in range(len(zone_labels)):
    for j in range(len(hour_labels)):
        val = data[i, j]
        text_color = "#0f1117" if val > 60 else WHITE
        ax.text(j, i, f"${val:.0f}", ha="center", va="center",
                fontsize=9, color=text_color, fontweight="bold")

# ── Highlight best cell ───────────────────────────────────────────────────
best_i, best_j = np.unravel_index(np.argmax(data), data.shape)
ax.add_patch(plt.Rectangle(
    (best_j - 0.5, best_i - 0.5), 1, 1,
    fill=False, edgecolor=TEAL, linewidth=2.5
))

ax.set_xticks(range(len(hour_labels)))
ax.set_xticklabels(hour_labels, fontsize=9)
ax.set_yticks(range(len(zone_labels)))
ax.set_yticklabels(zone_labels, fontsize=9)
ax.set_title("Avg Revenue Per Trip — Zone × Hour Window (2023–2025)")

cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label("Avg Revenue / Trip ($)", fontsize=9)
cbar.ax.yaxis.set_tick_params(color=WHITE)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=WHITE)

plt.tight_layout()
plt.show()